# Task 8: App Data Package and Prediction Module

This notebook builds the production data assets and standalone prediction module for the PreambleMap web application:
1. **Region Data Integration**:
   - Downloads and extracts Natural Earth 50m cultural vectors (`ne_50m_admin_0_countries.zip`).
   - Aligns all 193 constitutional records via exact name matching and verified aliases (`Antigua and Barbuda`, `Saint Kitts and Nevis`, `Marshall Islands`).
2. **`data/app/countries.csv`**:
   - Master country-level dataset (193 rows) with cluster assignments, labels, confidence scores, boundary flags, opening types, continent metadata, verbatim texts, provenance notes, and 5 nearest clustered neighbors by cosine similarity on $C_c$.
3. **`data/app/terms.csv`**:
   - Multi-faceted vocabulary package containing top 15 unigram c-TF-IDF, top 10 bigram c-TF-IDF, and top 12 document contrast terms per cluster, with generic term annotations and removal of token `"ii"`.
4. **`data/app/clusters.csv`**:
   - Cluster profile table containing human-readable labels, verbatim thematic descriptions, rule strength ratings, cohesion tiers, validation metrics, contrast keywords, top bigrams, central preambles, boundary members, median word counts, and geographic distributions.
5. **`data/app/model_assets.npz`**:
   - Compressed archive containing country names, $C_c$ coordinate vectors, training column mean, normalized cluster centroids, and cluster IDs.
6. **`src/preamble_model.py` Module & Verification Tests**:
   - End-to-end testing of `load_assets`, `embed_text`, and `classify_text`.
   - Vector reproducibility verification (maximum absolute error $< 10^{-4}$).
   - Centroid assignment consistency across the 158 clustered countries.
   - Inference benchmarking on test preambles (US constitutional preamble and Islamic republic formula).
   - Process resident memory (RSS) and latency profiling.


## 1. Input Verification and Geographic Data Integration (Natural Earth 50m)

Load all source data files, construct the centered and L2-normalized representations $C_c$, and match all 193 constitutional countries against Natural Earth 50m cultural vectors (`ne_50m_admin_0_countries.zip`).

In [1]:
import os
import re
import time
import psutil
import numpy as np
import pandas as pd
import geopandas as gpd
from sklearn.feature_extraction.text import CountVectorizer

# Environment paths
data_dir = '../data/processed'
app_dir = '../data/app'
ext_dir = '../data/external'
os.makedirs(app_dir, exist_ok=True)
os.makedirs(os.path.join(ext_dir, 'ne_50m_admin_0_countries'), exist_ok=True)

# Set offline environment for model loading
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
if not os.path.exists(os.path.expanduser('~/hf_cache')) and os.path.exists('/tmp/hf_cache'):
    os.environ['HF_HOME'] = '/tmp/hf_cache'

# 1. Load inputs
order_df = pd.read_csv(os.path.join(data_dir, 'final_country_order.csv'))
final_df = pd.read_csv(os.path.join(data_dir, 'cluster_final.csv'))
preamble_df = pd.read_csv(os.path.join(data_dir, 'preamble_final.csv'))
val_df = pd.read_csv(os.path.join(data_dir, 'label_validation.csv'))
emb_masked = np.load(os.path.join(data_dir, 'emb_final_masked.npy'))

# 2. Build C_c
col_mean = emb_masked.mean(axis=0).astype(np.float32)
centered = emb_masked - col_mean
norms = np.linalg.norm(centered, axis=1, keepdims=True)
C_c = (centered / norms).astype(np.float32)

assert len(order_df) == 158
assert len(final_df) == 158
assert len(preamble_df) == 193
assert C_c.shape == (158, 768)
print(f"Inputs validated: 158 clustered countries, 193 total preambles, C_c shape {C_c.shape}.")

# 3. Match 193 countries against Natural Earth 50m cultural vectors
shp_path = os.path.join(ext_dir, 'ne_50m_admin_0_countries', 'ne_50m_admin_0_countries.shp')
if not os.path.exists(shp_path):
    import zipfile
    with zipfile.ZipFile('/tmp/ne_50m_admin_0_countries.zip', 'r') as z:
        z.extractall(os.path.join(ext_dir, 'ne_50m_admin_0_countries'))

world_50m = gpd.read_file(shp_path)
name_to_cont = dict(zip(world_50m['NAME'], world_50m['CONTINENT']))

# Initial exact match
exact_matches = {c: name_to_cont[c] for c in preamble_df['Country'] if c in name_to_cont}
unmatched_initial = [c for c in preamble_df['Country'] if c not in name_to_cont]
print(f"\nNatural Earth exact matches: {len(exact_matches)} / 193")
print(f"Initially unmatched ({len(unmatched_initial)}): {unmatched_initial}")

# Search for unmatched candidates
print("\nCandidate matches in Natural Earth 50m:")
aliases = {}
for term, orig_name in [('Antigua', 'Antigua and Barbuda'), ('Kitts', 'Saint Kitts and Nevis'), ('Marshall', 'Marshall Islands')]:
    mask = pd.Series(False, index=world_50m.index)
    for col in ['NAME', 'NAME_LONG', 'ADMIN', 'FORMAL_EN']:
        mask = mask | world_50m[col].astype(str).str.contains(term, case=False, na=False)
    matches_sub = world_50m[mask][['NAME', 'NAME_LONG', 'ADMIN', 'FORMAL_EN', 'CONTINENT']]
    print(f"\nSearch for '{term}': {len(matches_sub)} row(s)")
    print(matches_sub.to_string(index=False))
    if len(matches_sub) == 1:
        matched_name = matches_sub['NAME'].iloc[0]
        aliases[orig_name] = matched_name

print(f"\nAdded aliases: {aliases}")

country_to_continent = {}
for c in preamble_df['Country']:
    target = aliases.get(c, c)
    country_to_continent[c] = name_to_cont.get(target, None)

unmatched_final = [c for c, cont in country_to_continent.items() if cont is None]
print(f"\nFinal continent match summary: {len(preamble_df) - len(unmatched_final)} matched, {len(unmatched_final)} unmatched.")
print(f"Unmatched countries list: {unmatched_final}")


Inputs validated: 158 clustered countries, 193 total preambles, C_c shape (158, 768).

Natural Earth exact matches: 190 / 193
Initially unmatched (3): ['Antigua and Barbuda', 'Marshall Islands', 'Saint Kitts and Nevis']

Candidate matches in Natural Earth 50m:

Search for 'Antigua': 1 row(s)
             NAME           NAME_LONG               ADMIN           FORMAL_EN     CONTINENT
Antigua and Barb. Antigua and Barbuda Antigua and Barbuda Antigua and Barbuda North America

Search for 'Kitts': 1 row(s)
               NAME             NAME_LONG                 ADMIN                           FORMAL_EN     CONTINENT
St. Kitts and Nevis Saint Kitts and Nevis Saint Kitts and Nevis Federation of Saint Kitts and Nevis North America

Search for 'Marshall': 1 row(s)
        NAME        NAME_LONG            ADMIN                        FORMAL_EN CONTINENT
Marshall Is. Marshall Islands Marshall Islands Republic of the Marshall Islands   Oceania

Added aliases: {'Antigua and Barbuda': 'Antigua and

## 2. Generating `data/app/countries.csv`

Construct the comprehensive master country dataset for the application:
- `Country`: Formal country name.
- `category`: Categorization (`clustered` [158], `no_preamble_in_dataset` [29], `formula_only` [6]).
- `final_cluster`, `cluster_label`: Assigned cluster index and label (blank for unclustered).
- `conf_sub`, `boundary_flag`, `runnerup_cluster`, `runnerup_label`, `runnerup_score`: Subsample stability profile.
- `extracted_words`, `opening_type`, `continent`, `preamble_text`: Content and structural attributes.
- `note`: Explanatory note for unclustered preambles.
- `source_note`: Specific provenance notices for historical texts (Magna Carta for UK, Third All-Russian Congress of Soviets for Russia).
- `neighbor_1` .. `neighbor_5` with `sim_1` .. `sim_5`: 5 nearest clustered countries by cosine similarity on $C_c$.

In [2]:
# 1. Labels dictionary
LABELS = {
    0: "Generic Rights & Freedoms (catch-all)",
    1: "African Charter & Unity",
    2: "Statehood & History",
    3: "Constituent Assembly & God",
    4: "Commonwealth 'Whereas'",
    5: "Islamic & Arab Identity",
    6: "Struggle & Socialism (mostly Asia)"
}

# 2. Source text verification for UK and Russia
uk_text = preamble_df[preamble_df['Country'] == 'United Kingdom']['extracted_text'].iloc[0]
ru_text = preamble_df[preamble_df['Country'] == 'Russia']['extracted_text'].iloc[0]

print("=== PROVENANCE TEXT VERIFICATION (FIRST 300 CHARACTERS) ===")
print(f"\nUnited Kingdom:\n'{str(uk_text)[:300]}'")
print(f"\nRussia:\n'{str(ru_text)[:300]}'")

# 3. Compute 5 nearest neighbors on C_c for all 158 clustered countries
sim_matrix = C_c @ C_c.T
np.fill_diagonal(sim_matrix, -999.0)
top5_indices = np.argsort(sim_matrix, axis=1)[:, ::-1][:, :5]

neighbors_map = {}
for i, c_name in enumerate(order_df['Country']):
    entry = {}
    for rk in range(5):
        j = top5_indices[i, rk]
        entry[f'neighbor_{rk+1}'] = order_df.loc[j, 'Country']
        entry[f'sim_{rk+1}'] = float(sim_matrix[i, j])
    neighbors_map[c_name] = entry

# 4. Map cluster_final data
final_by_country = {r['Country']: r for _, r in final_df.iterrows()}

countries_records = []
for idx, r in preamble_df.iterrows():
    c_name = r['Country']
    st = r['status']
    raw_text = str(r['extracted_text']) if pd.notna(r['extracted_text']) else ''
    
    if st in ['preamble_ok', 'preamble_cut']:
        cat = 'clustered'
    elif st == 'no_preamble_in_dataset':
        cat = 'no_preamble_in_dataset'
    elif st == 'formula_only':
        cat = 'formula_only'
    else:
        raise ValueError(f"Unknown status: {st}")
        
    continent_val = country_to_continent.get(c_name, '')
    
    # Note
    note_val = ''
    if cat == 'no_preamble_in_dataset':
        note_val = "The dataset text for this country is the body of the constitution with no preamble; the real constitution may still have one."
    elif cat == 'formula_only':
        note_val = "The preamble in the dataset is only an enacting formula, too short to cluster."
        
    # Source note
    src_note_val = ''
    if c_name == 'United Kingdom' and raw_text.startswith('Magna Carta'):
        src_note_val = "Dataset text is Magna Carta, not a constitutional preamble"
    elif c_name == 'Russia' and raw_text.startswith('The declaration of rights of the laboring and exploited people'):
        src_note_val = "Dataset text refers to the Third All-Russian Congress of Soviets (an early Soviet-era text), not the current preamble"
        
    # Preamble text
    pt_val = '' if cat == 'no_preamble_in_dataset' else raw_text
    
    if cat == 'clustered':
        c_info = final_by_country[c_name]
        fc = int(c_info['final_cluster'])
        cl_label = LABELS[fc]
        conf_sub = float(c_info['conf_sub'])
        b_flag = bool(c_info['boundary_flag'])
        ru_c = int(c_info['runnerup_sub'])
        ru_label = LABELS[ru_c]
        ru_score = float(c_info['runnerup_sub_score'])
        words = int(c_info['extracted_words'])
        op_type = str(c_info['opening_type'])
        
        n_info = neighbors_map[c_name]
        n1, s1 = n_info['neighbor_1'], n_info['sim_1']
        n2, s2 = n_info['neighbor_2'], n_info['sim_2']
        n3, s3 = n_info['neighbor_3'], n_info['sim_3']
        n4, s4 = n_info['neighbor_4'], n_info['sim_4']
        n5, s5 = n_info['neighbor_5'], n_info['sim_5']
    else:
        fc = ''
        cl_label = ''
        conf_sub = ''
        b_flag = ''
        ru_c = ''
        ru_label = ''
        ru_score = ''
        words = int(r['extracted_words']) if pd.notna(r['extracted_words']) else 0
        op_type = ''
        n1, s1, n2, s2, n3, s3, n4, s4, n5, s5 = '', '', '', '', '', '', '', '', '', ''
        
    countries_records.append({
        'Country': c_name,
        'category': cat,
        'final_cluster': fc,
        'cluster_label': cl_label,
        'conf_sub': conf_sub,
        'boundary_flag': b_flag,
        'runnerup_cluster': ru_c,
        'runnerup_label': ru_label,
        'runnerup_score': ru_score,
        'extracted_words': words,
        'opening_type': op_type,
        'continent': continent_val,
        'preamble_text': pt_val,
        'note': note_val,
        'source_note': src_note_val,
        'neighbor_1': n1,
        'sim_1': s1,
        'neighbor_2': n2,
        'sim_2': s2,
        'neighbor_3': n3,
        'sim_3': s3,
        'neighbor_4': n4,
        'sim_4': s4,
        'neighbor_5': n5,
        'sim_5': s5
    })

df_countries = pd.DataFrame(countries_records)

# Assertions
assert len(df_countries) == 193, f"Expected 193 rows, got {len(df_countries)}"
assert (df_countries['category'] == 'clustered').sum() == 158
assert (df_countries['category'] == 'no_preamble_in_dataset').sum() == 29
assert (df_countries['category'] == 'formula_only').sum() == 6

clustered_check = df_countries[df_countries['category'] == 'clustered'].set_index('Country')
final_check = final_df.set_index('Country')
assert (clustered_check['final_cluster'].astype(int) == final_check['final_cluster']).all()

countries_csv_path = os.path.join(app_dir, 'countries.csv')
df_countries.to_csv(countries_csv_path, index=False)
print(f"\nSuccessfully written {countries_csv_path} ({df_countries.shape[0]} rows, {df_countries.shape[1]} cols).")
print(f"Category breakdown:\n{df_countries['category'].value_counts().to_string()}")


=== PROVENANCE TEXT VERIFICATION (FIRST 300 CHARACTERS) ===

United Kingdom:
'Magna Carta THE GREAT CHARTER OF THE LIBERTIES OF ENGLAND, AND OF THE LIBERTIES OF THE FOREST; CONFIRMED BY KING EDWARD, IN THE TWENTY-FIFTH YEAR OF HIS REIGN. EDWARD by the Grace of God King of England, Lord of Ireland, and Duke of Guyan, to all Archbishops, Bishops, c. We have seen the Great Chart'

Russia:
'The declaration of rights of the laboring and exploited people approved by the Third All-Russian Congress of Soviets in January, , together with the constitution of the Soviet Republic, approved by the Fifth Congress, constitutes a single fundamental law of the Russian Socialist Federated Soviet Rep'

Successfully written ../data/app/countries.csv (193 rows, 25 cols).
Category breakdown:
category
clustered                 158
no_preamble_in_dataset     29
formula_only                6


## 3. Generating `data/app/terms.csv`

Extract the top vocabulary terms per cluster across three complementary methodologies:
1. **Unigram c-TF-IDF** (top 15 terms per cluster).
2. **Bigram c-TF-IDF** (top 10 bigrams per cluster).
3. **Document Contrast** (top 12 unigrams per cluster with $\text{share}_{\text{in}} \ge 0.30$ and ratio $\ge 2.0$).

Filter out token `"ii"` from Cluster 6 lists (originating as an artifact from roman numeral headings). Flag generic constitutional terms (`constitution`, `people`, `peoples`, `state`, `national`, `shall`, `following`, `republic`, `nation`).

In [3]:
clusterable = preamble_df[preamble_df['status'].isin(['preamble_ok', 'preamble_cut'])].copy().reset_index(drop=True)

def mask_country_name(country, text):
    words = re.findall(r'[a-zA-Z]+', country)
    cap_words = [w for w in words if w and w[0].isupper() and len(w) >= 4]
    terms = set([country] + cap_words)
    sorted_terms = sorted(terms, key=len, reverse=True)
    pattern = r'\b(?:' + '|'.join(re.escape(t) for t in sorted_terms) + r')\b'
    masked_text, count = re.subn(pattern, '[COUNTRY]', text, flags=re.IGNORECASE)
    return masked_text, count

cleaned_texts = []
for idx, r in clusterable.iterrows():
    c = r['Country']
    raw_t = ' '.join(str(r['extracted_text']).split())
    m_t, _ = mask_country_name(c, raw_t)
    no_country = re.sub(r'\[COUNTRY\]', ' ', m_t)
    cleaned_texts.append(' '.join(no_country.split()))

clusterable['cleaned_text'] = cleaned_texts
clusterable['final_cluster'] = final_df['final_cluster']

# Concatenated documents per cluster
cluster_docs = []
for c in range(7):
    c_texts = clusterable[clusterable['final_cluster'] == c]['cleaned_text'].tolist()
    cluster_docs.append(' '.join(c_texts))

# Binary document occurrence matrices
cv_doc_uni = CountVectorizer(stop_words='english', binary=True)
doc_term_uni = cv_doc_uni.fit_transform(clusterable['cleaned_text']).toarray()
vocab_doc_uni = cv_doc_uni.get_feature_names_out()

cv_doc_bi = CountVectorizer(ngram_range=(2, 2), stop_words='english', binary=True)
doc_term_bi = cv_doc_bi.fit_transform(clusterable['cleaned_text']).toarray()
vocab_doc_bi = cv_doc_bi.get_feature_names_out()

# c-TF-IDF Unigrams
cv_uni = CountVectorizer(stop_words='english', min_df=3)
X_uni = cv_uni.fit_transform(cluster_docs).toarray()
vocab_uni = cv_uni.get_feature_names_out()
A_uni = X_uni.sum(axis=1).mean()
f_t_uni = X_uni.sum(axis=0)
c_tfidf_uni = X_uni * np.log(1.0 + A_uni / f_t_uni)

# c-TF-IDF Bigrams
cv_bi = CountVectorizer(ngram_range=(2, 2), stop_words='english', min_df=3)
X_bi = cv_bi.fit_transform(cluster_docs).toarray()
vocab_bi = cv_bi.get_feature_names_out()
A_bi = X_bi.sum(axis=1).mean()
f_t_bi = X_bi.sum(axis=0)
c_tfidf_bi = X_bi * np.log(1.0 + A_bi / f_t_bi)

generic_set = {'constitution', 'people', 'peoples', 'state', 'national', 'shall', 'following', 'republic', 'nation'}

terms_records = []
for c in range(7):
    in_mask = (clusterable['final_cluster'] == c).values
    out_mask = ~in_mask
    n_in = in_mask.sum()
    n_out = out_mask.sum()
    
    # 1. Unigrams c-TF-IDF (top 15, dropping 'ii' for cluster 6)
    top_u_idx = np.argsort(c_tfidf_uni[c])[::-1]
    rank_u = 1
    for idx in top_u_idx:
        term = vocab_uni[idx]
        if c == 6 and term == 'ii':
            continue
        score = float(c_tfidf_uni[c, idx])
        col_idx = np.where(vocab_doc_uni == term)[0]
        s_in = float(doc_term_uni[in_mask, col_idx[0]].sum() / n_in) if len(col_idx) > 0 else 0.0
        s_out = float(doc_term_uni[out_mask, col_idx[0]].sum() / n_out) if len(col_idx) > 0 else 0.0
        
        terms_records.append({
            'cluster': c,
            'kind': 'unigram_ctfidf',
            'rank': rank_u,
            'term': term,
            'score': score,
            'share_in': s_in,
            'share_out': s_out,
            'generic': term in generic_set
        })
        rank_u += 1
        if rank_u > 15:
            break
            
    # 2. Bigrams c-TF-IDF (top 10)
    top_b_idx = np.argsort(c_tfidf_bi[c])[::-1]
    rank_b = 1
    for idx in top_b_idx:
        term = vocab_bi[idx]
        score = float(c_tfidf_bi[c, idx])
        col_idx = np.where(vocab_doc_bi == term)[0]
        s_in = float(doc_term_bi[in_mask, col_idx[0]].sum() / n_in) if len(col_idx) > 0 else 0.0
        s_out = float(doc_term_bi[out_mask, col_idx[0]].sum() / n_out) if len(col_idx) > 0 else 0.0
        
        terms_records.append({
            'cluster': c,
            'kind': 'bigram_ctfidf',
            'rank': rank_b,
            'term': term,
            'score': score,
            'share_in': s_in,
            'share_out': s_out,
            'generic': any(w in generic_set for w in term.split())
        })
        rank_b += 1
        if rank_b > 10:
            break
            
    # 3. Document Contrast (top 12)
    shares_in_all = doc_term_uni[in_mask].sum(axis=0) / n_in
    shares_out_all = doc_term_uni[out_mask].sum(axis=0) / n_out
    ratios = shares_in_all / np.maximum(shares_out_all, 0.01)
    
    valid_idx = np.where((shares_in_all >= 0.30) & (ratios >= 2.0))[0]
    sorted_valid = sorted(valid_idx, key=lambda i: (shares_in_all[i], ratios[i]), reverse=True)
    
    rank_cnt = 1
    for idx in sorted_valid:
        term = vocab_doc_uni[idx]
        if c == 6 and term == 'ii':
            continue
        s_in = float(shares_in_all[idx])
        s_out = float(shares_out_all[idx])
        score = float(ratios[idx])
        
        terms_records.append({
            'cluster': c,
            'kind': 'contrast',
            'rank': rank_cnt,
            'term': term,
            'score': score,
            'share_in': s_in,
            'share_out': s_out,
            'generic': term in generic_set
        })
        rank_cnt += 1
        if rank_cnt > 12:
            break

df_terms = pd.DataFrame(terms_records)

# Assertions on spot check values
row_c1 = df_terms[(df_terms['cluster'] == 1) & (df_terms['term'] == 'african')].iloc[0]
assert np.isclose(row_c1['share_in'], 0.769, atol=0.001), f"Expected 0.769, got {row_c1['share_in']}"
assert np.isclose(row_c1['share_out'], 0.053, atol=0.001), f"Expected 0.053, got {row_c1['share_out']}"

row_c5 = df_terms[(df_terms['cluster'] == 5) & (df_terms['term'] == 'merciful')].iloc[0]
assert np.isclose(row_c5['share_in'], 0.588, atol=0.001), f"Expected 0.588, got {row_c5['share_in']}"

row_c6 = df_terms[(df_terms['cluster'] == 6) & (df_terms['term'] == 'socialist')].iloc[0]
assert np.isclose(row_c6['share_in'], 0.467, atol=0.001), f"Expected 0.467, got {row_c6['share_in']}"

# Verify 'ii' dropped from cluster 6
c6_terms = df_terms[df_terms['cluster'] == 6]['term'].tolist()
assert 'ii' not in c6_terms, "Found token 'ii' in cluster 6 terms!"

terms_csv_path = os.path.join(app_dir, 'terms.csv')
df_terms.to_csv(terms_csv_path, index=False)
print(f"Successfully written {terms_csv_path} ({df_terms.shape[0]} rows, {df_terms.shape[1]} cols).")
print(f"Verified spot checks: cluster 1 'african' (0.769/0.053), cluster 5 'merciful' (0.588), cluster 6 'socialist' (0.467).")


Successfully written ../data/app/terms.csv (240 rows, 8 cols).
Verified spot checks: cluster 1 'african' (0.769/0.053), cluster 5 'merciful' (0.588), cluster 6 'socialist' (0.467).


## 4. Generating `data/app/clusters.csv`

Assemble the cluster metadata package:
- `cluster`: Integer cluster identifier (0 to 6).
- `label`: Standardized human-readable label.
- `n`: Cluster size.
- `description`: Verbatim thematic description.
- `rule_rating`: Rule-based lexical strength rating (`strong` for cluster 5, `moderate` for others).
- `cohesion_tier`: Vector space clustering cohesion tier (`well-defined` for 1, 2, 3, 5; `loosely defined` for 0, 4, 6).
- `coverage`, `precision`, `mean_silhouette`, `mean_conf_sub`: Metrics from `label_validation.csv`.
- `top_contrast_terms`: Up to 8 non-generic contrast unigrams joined by `"; "`.
- `top_bigrams`: Top 5 characteristic bigrams joined by `"; "`.
- `central_countries`: 5 countries closest to cluster centroid on $C_c$.
- `boundary_countries`: Subsample boundary cases flagged within the cluster.
- `median_words`: Median word count of member preambles.
- `continents`: Continent counts joined descending by count.
- `strength_note`: Explanatory methodology note.

In [4]:
descriptions = {
    0: "A broad catch-all. 22 of 30 preambles open 'We the people' and speak of justice, freedom and God (USA, India, Ghana, Kenya, Mongolia, Chile), but most 'We the people' preambles (40 of 62) sit in other clusters. Its words are generic, so treat it as a default group, not a distinct theme.",
    1: "All 26 members are in Africa. Preambles cite the African Charter, the Universal Declaration and the 'rights of man', and stress national and African unity (Senegal, Burkina Faso, Guinea, Mali, Benin).",
    2: "Central and Eastern Europe and the former USSR, plus Germany, Switzerland and Andorra. Preambles stress statehood, centuries of history, responsibility to future generations and self-determination (Montenegro, Belarus, Croatia, Slovenia, Czechia).",
    3: "Ibero-American tradition. Preambles invoke God's protection, a constituent assembly and a decree (Venezuela, Dominican Republic, Colombia, Peru, Spain, Portugal), plus Lusophone Africa.",
    4: "Mostly Caribbean, Pacific and Commonwealth states (Bahamas, Barbados, Solomon Islands, Canada, Australia, UK). 12 of 22 open 'Whereas' and stress the supremacy of God, free institutions and the dignity of the human person. Loosely defined: 6 of 22 members use none of the cluster's typical vocabulary (including France and the UK), and many are island states.",
    5: "Preambles open 'In the name of God, the Merciful' (10 of 17) and stress Islam, Arab identity, revolution and social justice (Kuwait, Iran, Iraq, Syria, Egypt). The most cohesive cluster, including Pakistan, Turkey, Afghanistan, Brunei and the Maghreb states.",
    6: "Mostly Asian states whose preambles recount a struggle for independence (80% of members) and often mention socialism (53%) or party politics (53%): Laos, China, Bangladesh, Sri Lanka, Nepal, Vietnam, North Korea. Loosely defined: Japan and South Sudan sit on the boundary."
}

# Attach filled continent to final_df
final_df['continent_50m'] = [country_to_continent[c] for c in final_df['Country']]

strength_note_text = "Rule rating: strong if coverage >= 60% and precision >= 50%, weak if coverage < 40%, otherwise moderate. Loosely defined means the cluster is among the three with the lowest mean silhouette and lowest mean subsample confidence."

# Central countries
central_countries_str = {}
for c in range(7):
    idx_c = np.where(final_df['final_cluster'] == c)[0]
    centroid = C_c[idx_c].mean(axis=0, keepdims=True)
    centroid = centroid / np.linalg.norm(centroid)
    sims = (C_c[idx_c] @ centroid.T).ravel()
    top5_idx = idx_c[np.argsort(sims)[::-1][:5]]
    top5_names = order_df.loc[top5_idx, 'Country'].tolist()
    central_countries_str[c] = '; '.join(top5_names)

cluster_rows = []
for c in range(7):
    v_row = val_df[val_df['cluster'] == c].iloc[0]
    n_c = int(v_row['n'])
    cov = float(v_row['coverage'])
    prec = float(v_row['precision'])
    mean_sil = float(v_row['mean_silhouette'])
    mean_conf_sub = float(v_row['mean_conf_sub'])
    
    rule_rating = "strong" if c == 5 else "moderate"
    cohesion_tier = "well-defined" if c in [1, 2, 3, 5] else "loosely defined"
    
    # Top 8 non-generic contrast terms
    c_contrast = df_terms[(df_terms['cluster'] == c) & (df_terms['kind'] == 'contrast') & (~df_terms['generic'])]
    top_contrast_list = c_contrast['term'].head(8).tolist()
    top_contrast_str = '; '.join(top_contrast_list)
    
    # Top 5 bigrams
    c_bigrams = df_terms[(df_terms['cluster'] == c) & (df_terms['kind'] == 'bigram_ctfidf')]
    top_bigrams_list = c_bigrams['term'].head(5).tolist()
    top_bigrams_str = '; '.join(top_bigrams_list)
    
    # Boundary countries
    b_countries = final_df[(final_df['final_cluster'] == c) & (final_df['boundary_flag'])]['Country'].tolist()
    boundary_str = '; '.join(b_countries)
    
    # Median words
    med_words = float(final_df[final_df['final_cluster'] == c]['extracted_words'].median())
    
    # Continents
    cont_counts = final_df[final_df['final_cluster'] == c]['continent_50m'].value_counts()
    continents_str = '; '.join([f"{k} {v}" for k, v in cont_counts.items()])
    
    cluster_rows.append({
        'cluster': c,
        'label': LABELS[c],
        'n': n_c,
        'description': descriptions[c],
        'rule_rating': rule_rating,
        'cohesion_tier': cohesion_tier,
        'coverage': cov,
        'precision': prec,
        'mean_silhouette': mean_sil,
        'mean_conf_sub': mean_conf_sub,
        'top_contrast_terms': top_contrast_str,
        'top_bigrams': top_bigrams_str,
        'central_countries': central_countries_str[c],
        'boundary_countries': boundary_str,
        'median_words': med_words,
        'continents': continents_str,
        'strength_note': strength_note_text
    })

df_clusters = pd.DataFrame(cluster_rows)
clusters_csv_path = os.path.join(app_dir, 'clusters.csv')
df_clusters.to_csv(clusters_csv_path, index=False)
print(f"Successfully written {clusters_csv_path} ({df_clusters.shape[0]} rows, {df_clusters.shape[1]} cols).")


Successfully written ../data/app/clusters.csv (7 rows, 17 cols).


## 5. Exporting `data/app/model_assets.npz`

Package embedding model vectors and centroid coordinates into a compressed NumPy archive:
- `country_names`: 158 country names in row order.
- `vectors`: $158 \times 768$ float32 array ($C_c$).
- `column_mean`: 768-dimensional float32 training column mean vector.
- `centroids`: $7 \times 768$ float32 normalized cluster centroid vectors.
- `cluster_ids`: 158-element integer array of final cluster assignments.

In [5]:
country_names_arr = np.array(order_df['Country'], dtype='str')
vectors_arr = C_c.astype(np.float32)
col_mean_arr = col_mean.astype(np.float32)

centroids_arr = np.zeros((7, 768), dtype=np.float32)
for c in range(7):
    c_mean = C_c[final_df['final_cluster'] == c].mean(axis=0)
    centroids_arr[c] = (c_mean / np.linalg.norm(c_mean)).astype(np.float32)

cluster_ids_arr = final_df['final_cluster'].values.astype(np.int32)

assets_path = os.path.join(app_dir, 'model_assets.npz')
np.savez(
    assets_path,
    country_names=country_names_arr,
    vectors=vectors_arr,
    column_mean=col_mean_arr,
    centroids=centroids_arr,
    cluster_ids=cluster_ids_arr
)

# Verify loading without pickle
loaded_assets = np.load(assets_path)
assert loaded_assets['country_names'].shape == (158,)
assert loaded_assets['vectors'].shape == (158, 768)
assert loaded_assets['column_mean'].shape == (768,)
assert loaded_assets['centroids'].shape == (7, 768)
assert loaded_assets['cluster_ids'].shape == (158,)
print(f"Successfully saved and verified {assets_path}.")
for k in loaded_assets.files:
    print(f"  {k:<15}: shape {loaded_assets[k].shape}, dtype {loaded_assets[k].dtype}")


Successfully saved and verified ../data/app/model_assets.npz.
  country_names  : shape (158,), dtype <U24
  vectors        : shape (158, 768), dtype float32
  column_mean    : shape (768,), dtype float32
  centroids      : shape (7, 768), dtype float32
  cluster_ids    : shape (158,), dtype int32


## 6. Standalone Prediction Module (`src/preamble_model.py`)

Verify the modular prediction library `src/preamble_model.py` containing:
- `load_assets(path=None)`
- `embed_text(model, text, mask_name=None, column_mean=None)`
- `classify_text(model, assets, text, mask_name=None, top_n=5)`

In [6]:
import sys
sys.path.append('..')
import src.preamble_model as pm

print("preamble_model functions:")
for fn_name in ['load_assets', 'embed_text', 'classify_text']:
    fn = getattr(pm, fn_name, None)
    assert fn is not None, f"Function {fn_name} missing from src/preamble_model.py"
    print(f"  - {fn_name}: {fn}")


preamble_model functions:
  - load_assets: <function load_assets at 0x7f90baa405e0>
  - embed_text: <function embed_text at 0x7f90baa40680>
  - classify_text: <function classify_text at 0x7f90baa40720>


## 7. Model Validation Tests

Execute four quality assurance tests:
1. **Reproducibility**: Compare `embed_text` against stored $C_c$ for 10 countries (5 longest chunk counts + 5 random seeds).
2. **Nearest Centroid Consistency**: Validate cluster assignment alignment across all 158 countries against nearest centroids.
3. **Inference Examples**: Classify the US constitutional preamble and an Islamic republic preamble test text.
4. **Latency & Memory Profiling**: Measure inference time and process resident memory (RSS in MB) before and after model instantiation.

In [7]:
# 1. Process memory before model loading
process = psutil.Process(os.getpid())
rss_before_mb = process.memory_info().rss / (1024.0 * 1024.0)
print(f"Process Resident Memory (RSS) before model load: {rss_before_mb:.2f} MB")

from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-mpnet-base-v2')

rss_after_mb = process.memory_info().rss / (1024.0 * 1024.0)
print(f"Process Resident Memory (RSS) after model load:  {rss_after_mb:.2f} MB")
print(f"Model Memory Overhead:                          {rss_after_mb - rss_before_mb:.2f} MB")

assets = pm.load_assets(assets_path)

# --- Test 7a: Reproducibility ---
# Chunk count calculation
chunk_counts = []
for idx, r in clusterable.iterrows():
    c = r['Country']
    m_t, _ = pm.mask_country_name(c, r['extracted_text'])
    ch, _ = pm.chunk_variant_a(m_t, model.tokenizer)
    chunk_counts.append((idx, c, len(ch)))

chunk_counts_sorted = sorted(chunk_counts, key=lambda x: x[2], reverse=True)
top5_chunks_idx = [x[0] for x in chunk_counts_sorted[:5]]

rng = np.random.RandomState(42)
other_idx = [i for i in range(158) if i not in top5_chunks_idx]
random5_idx = rng.choice(other_idx, size=5, replace=False).tolist()

test_10_indices = top5_chunks_idx + random5_idx

print("\n=== TEST 7A: REPRODUCIBILITY (10 COUNTRIES) ===")
max_abs_diffs = []
for idx in test_10_indices:
    c_name = clusterable.loc[idx, 'Country']
    raw_t = clusterable.loc[idx, 'extracted_text']
    stored_vec = assets['vectors'][idx]
    
    recomputed_vec = pm.embed_text(model, raw_t, mask_name=c_name, column_mean=assets['column_mean'])
    diff = float(np.max(np.abs(recomputed_vec - stored_vec)))
    max_abs_diffs.append((c_name, diff))
    print(f"  {c_name:<25}: max abs diff = {diff:.6e}")

overall_max_diff = max(d[1] for d in max_abs_diffs)
print(f"\nMaximum absolute difference across all 10 countries: {overall_max_diff:.6e}")
assert overall_max_diff < 1e-4, f"Reproducibility assertion failed: {overall_max_diff} >= 1e-4"
print("Assertion Passed: All differences are strictly below 1e-4.")

# --- Test 7b: Nearest Centroid Consistency ---
print("\n=== TEST 7B: NEAREST CENTROID ALIGNMENT (ALL 158 COUNTRIES) ===")
sims_all = assets['vectors'] @ assets['centroids'].T
nearest_centroids = np.argmax(sims_all, axis=1)

matches = (nearest_centroids == assets['cluster_ids'])
match_frac = float(matches.mean())
print(f"Fraction whose nearest centroid equals final_cluster: {matches.sum()}/158 ({match_frac:.4f} or {match_frac*100:.2f}%)")

diff_idx = np.where(~matches)[0]
print(f"Countries where nearest centroid differs: {len(diff_idx)}")
for idx in diff_idx:
    c_name = assets['country_names'][idx]
    fc = assets['cluster_ids'][idx]
    nc = nearest_centroids[idx]
    sim_fc = float(sims_all[idx, fc])
    sim_nc = float(sims_all[idx, nc])
    print(f"  {c_name:<25}: final_cluster = {fc} ({sim_fc:.4f}), nearest_centroid = {nc} ({sim_nc:.4f})")

# --- Test 7c: classify_text on Test Preambles ---
print("\n=== TEST 7C: CLASSIFY_TEXT EXAMPLES ===")
text_us = "We the People of the United States, in Order to form a more perfect Union, establish Justice, insure domestic Tranquility, provide for the common defence, promote the general Welfare, and secure the Blessings of Liberty to ourselves and our Posterity, do ordain and establish this Constitution for the United States of America."
res_us = pm.classify_text(model, assets, text_us, mask_name="United States of America")

print("1. United States Constitutional Preamble:")
print("Top 2 Clusters:")
for cid, clab, sim in res_us['top_clusters']:
    print(f"  Cluster {cid} ({clab}): {sim:.4f}")
print("Top 5 Countries:")
for cname, sim in res_us['top_countries']:
    print(f"  {cname:<25}: {sim:.4f}")
print(f"Close gap flag (<0.03): {res_us['close_gap_flag']} (gap: {res_us['gap']:.4f})")

text_islamic = "In the name of God, the Merciful, the Compassionate, we, the people of this republic, proclaim our faith in Islam, social justice and Arab unity."
res_is = pm.classify_text(model, assets, text_islamic)

print("\n2. Islamic Republic Preamble:")
print("Top 2 Clusters:")
for cid, clab, sim in res_is['top_clusters']:
    print(f"  Cluster {cid} ({clab}): {sim:.4f}")
print("Top 5 Countries:")
for cname, sim in res_is['top_countries']:
    print(f"  {cname:<25}: {sim:.4f}")
print(f"Close gap flag (<0.03): {res_is['close_gap_flag']} (gap: {res_is['gap']:.4f})")

# --- Test 7d: Latency Benchmarking ---
t0 = time.perf_counter()
_ = pm.classify_text(model, assets, text_us, mask_name="United States of America")
t1 = time.perf_counter()
latency_ms = (t1 - t0) * 1000.0
print(f"\n=== TEST 7D: INFERENCE TIMING ===")
print(f"Latency for one classify_text call: {latency_ms:.2f} ms")


Process Resident Memory (RSS) before model load: 300.47 MB


/home/devpanchal/Projects/PreambleMap/venv/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Token indices sequence length is longer than the specified maximum sequence length for this model (1501 > 384). Running this sequence through the model will result in indexing errors


Process Resident Memory (RSS) after model load:  587.42 MB
Model Memory Overhead:                          286.95 MB



=== TEST 7A: REPRODUCIBILITY (10 COUNTRIES) ===


...

  Niger                    : max abs diff = 0.000000e+00


  Seychelles               : max abs diff = 0.000000e+00


  Chad                     : max abs diff = 0.000000e+00

Maximum absolute difference across all 10 countries: 1.713634e-07
Assertion Passed: All differences are strictly below 1e-4.

=== TEST 7B: NEAREST CENTROID ALIGNMENT (ALL 158 COUNTRIES) ===
Fraction whose nearest centroid equals final_cluster: 158/158 (1.0000 or 100.00%)
Countries where nearest centroid differs: 0

=== TEST 7C: CLASSIFY_TEXT EXAMPLES ===
1. United States Constitutional Preamble:
Top 2 Clusters:
  Cluster 0 (Generic Rights & Freedoms (catch-all)): 0.5118
  Cluster 2 (Statehood & History): 0.0793
Top 5 Countries:
  United States of America : 0.9974
  Mongolia                 : 0.4823
  Ghana                    : 0.4469
  Tajikistan               : 0.3973
  Turkmenistan             : 0.3602
Close gap flag (<0.03): False (gap: 0.4326)



2. Islamic Republic Preamble:
Top 2 Clusters:
  Cluster 5 (Islamic & Arab Identity): 0.5184
  Cluster 0 (Generic Rights & Freedoms (catch-all)): 0.0157
Top 5 Countries:
  Egypt                    : 0.4676
  Kuwait                   : 0.4158
  Iraq                     : 0.3874
  United Arab Emirates     : 0.3857
  Iran                     : 0.3718
Close gap flag (<0.03): False (gap: 0.5027)



=== TEST 7D: INFERENCE TIMING ===
Latency for one classify_text call: 383.20 ms


## 8. Summary of Generated Assets and Git Status

Display the first 3 rows and file sizes for each exported application dataset.

In [8]:
print("=== FIRST 3 ROWS: countries.csv ===")
print(df_countries.head(3).to_string())

print("\n=== FIRST 3 ROWS: terms.csv ===")
print(df_terms.head(3).to_string())

print("\n=== FIRST 3 ROWS: clusters.csv ===")
print(df_clusters.head(3).to_string())

print("\n=== FILE SIZES ===")
files_to_check = [
    os.path.join(app_dir, 'countries.csv'),
    os.path.join(app_dir, 'terms.csv'),
    os.path.join(app_dir, 'clusters.csv'),
    os.path.join(app_dir, 'model_assets.npz'),
    '../src/preamble_model.py'
]

for fp in files_to_check:
    size_bytes = os.path.getsize(fp)
    if size_bytes >= 1024 * 1024:
        size_str = f"{size_bytes / (1024*1024):.2f} MB"
    else:
        size_str = f"{size_bytes / 1024:.2f} KB"
    print(f"  {os.path.basename(fp):<22}: {size_str:>10} ({size_bytes} bytes)")


=== FIRST 3 ROWS: countries.csv ===
       Country   category final_cluster            cluster_label  conf_sub boundary_flag runnerup_cluster                      runnerup_label runnerup_score  extracted_words    opening_type continent                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             